In [ ]:
import json
import time
import os
import kaggle_benchmarks as kbench

In [ ]:
@kbench.task(name=f"Summary events", description=f"Tóm lược nội dung (xử lý từng event)")
def summarize_task(llm) -> None:

    # Source transcripts đã được group event và sửa lỗi (nếu có) từ bước trước
    # Có thể làm tuần tự sau khi trích xuất keyword, hoặc chạy song song với bước trích xuất keyword (nếu muốn)
    # ngoài ra nếu event có gắn các tag như "intro", "outro", "ads" thì có thể bỏ qua tóm lược các event này để tiết kiệm chi phí
    base_input_dir = f"./keyword_video_transcripts" | f"./fixed_video_transcripts"
    base_output_dir = "./summary_video_transcripts"
    target_path = f"{base_input_dir}/transcripts"

    all_json_paths = []
    if os.path.exists(target_path):
        for file in os.listdir(target_path):
            if file.endswith(".json"):
                all_json_paths.append(os.path.join(target_path, file))
    else:
        print(f"❌ Không tìm thấy thư mục: {target_path}. Hãy kiểm tra lại LEVEL_NUMBER!")
        return

    all_json_paths.sort()
    total_files = len(all_json_paths)

    print(f"Starting summarize_task. Tìm thấy tổng cộng {total_files} file JSON.")
    print("----------------------------------------------------------------------")

    for idx, input_path in enumerate(all_json_paths, 1):
        relative_path = os.path.relpath(input_path, base_input_dir)
        output_path = os.path.join(base_output_dir, relative_path)
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        print(f"🎬 [{idx}/{total_files}] Đang xử lý: {relative_path}")

        try:
            with open(input_path, "r", encoding="utf-8") as f:
                transcript = json.load(f)
        except Exception as e:
            print(f"    ❌ Lỗi đọc file: {e}. Bỏ qua file này.")
            continue

        all_events = transcript.get('events', [])
        total_events = len(all_events)

        if total_events == 0:
            print(f"    ⏩ File trống (0 events). Copy nguyên bản sang output.")
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(transcript, f, ensure_ascii=False, indent=2)
            continue

        print(f"    📦 Có {total_events} events. Bắt đầu xử lý từng event một...")

        for e_idx, ev in enumerate(all_events, 1):
            event_text = ev["text"].strip()
            topic = ""
            # Topic của mỗi video có thể khác nhau, check benchmark/prompts/map-video-topic.csv để xem chi tiết
            prompt = f"""Bạn là chuyên gia tóm lược cho video {topic}. Hãy tóm lược đoạn tin sau.

YÊU CẦU:
- Chỉ tóm tắt trong khoảng 3-5 câu.
- Không thêm giải thích, không đánh số thứ tự, không dùng markdown, không lặp lại nguyên văn cả câu gốc.

Đoạn tin: {event_text}"""

            max_retries = 5
            retry_count = 0
            success = False
            summary = ""

            while retry_count < max_retries and not success:
                try:
                    response_text = llm.prompt(prompt)
                    summary = response_text.strip()
                    success = True
                    time.sleep(6.0)

                except Exception as e:
                    retry_count += 1
                    print(f"    ⚠️ Lỗi event {ev.get('event_id', e_idx)} (Lần thử {retry_count}/{max_retries}): {e}")
                    if retry_count < max_retries:
                        wait_time = 20.0 * (2 ** (retry_count - 1))
                        print(f"    💤 Ngủ đông {wait_time:.0f} giây...")
                        time.sleep(wait_time)
                    else:
                        print(f"    ❌ Event {ev.get('event_id', e_idx)} thất bại sau {max_retries} lần. Để summary rỗng!")

            ev["summary"] = summary   # chỉ thêm field mới, giữ nguyên mọi field khác

            if e_idx % 20 == 0 or e_idx == total_events:
                print(f"    ...đã xử lý {e_idx}/{total_events} events")

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(transcript, f, ensure_ascii=False, indent=2)

        print(f"    ✅ Đã lưu file có summary.")

    print(f"Summary task completed. Tất cả {total_files} file đã được xử lý và lưu vào {base_output_dir}.")
    kbench.assertions.assert_true(True, expectation=f"Done with {total_files} files.")

# Run the summarize_task with the specified LLM
summarize_task.run(llm=kbench.llms["google/gemini-3.5-flash"])